# Amazon Top-20 | LightGCN·NGCF 대규모 Ablation + 1개 최종 제출

이 노트북 하나로 **팀 공통 Validation 검증 → 여러 모델/하이퍼파라미터 Ablation → Recall@20 순위 결정 → 1위 설정으로 전체 train 재학습 → submission.csv 생성/검사**까지 진행합니다.

**기존 연구 조건 유지**
- 팀 공통 `amazon_topk_teamkit_20261110.zip`의 Validation CSV와 분할 ID(`f0dfad2f4431a876`)는 변경하지 않습니다.
- 학습기는 앞서 실험한 **기존 `graph_models.py` 그대로** 사용합니다. 즉 한 Epoch당 Adam 업데이트 한 번인 원래 실험 설계를 유지합니다.
- 모든 실험은 동일한 Validation 사용자/정답, seed=42, Recall@20을 사용하며 **한 번에 하나의 조건만 바꾸는 ablation**과 일부 조건 조합 실험을 함께 합니다.
- `test.csv`의 정답은 사용하거나 접근하지 않습니다. Test는 **최종 1개 파일**을 만드는 데만 사용합니다.

**준비하기:** Kaggle `File → Import Notebook` → 오른쪽 Input에 대회 데이터 5개 + 이전에 업로드한 **Private 팀 키트 데이터셋** 추가 → Settings에서 GPU T4 → `Run All`.

**실행량 주의:** 기본 `RUN_PROFILE='MAX'`는 최대 19가지 실험을 순서대로 실행합니다. GPU 시간/메모리/세션 제한이 걱정되면 설정 셀에서 `BALANCED`(약 12가지) 또는 `QUICK`(약 5가지)로 바꿔 실행하세요. 마지막 제출 파일은 Kaggle에 자동 업로드되지 않으며, 직접 제출해야 합니다.

**최종 제출 주의:** Validation을 여러 번 보면서 튜닝하면 검증셋 과적합이 생길 수 있습니다. 리더보드 점수는 달라질 수 있습니다.

## 1. 환경과 GPU 확인

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import shutil
import subprocess
import sys
import time
import zipfile

import torch

WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)
OUT = WORK / 'ablation_top20'
OUT.mkdir(parents=True, exist_ok=True)
roots = sorted({p.parent for p in Path('/kaggle/input').rglob('train.csv')
                if (p.parent / 'test.csv').exists()})
if len(roots) != 1:
    raise RuntimeError(f'대회 train.csv/test.csv 데이터셋을 하나만 연결하세요. 현재: {roots}')
DATA = roots[0]
TRAIN = DATA / 'train.csv'
TEST = DATA / 'test.csv'
VALID = WORK / 'validation'
SEED = 42
DEVICE = 'cuda:0'
EXPECTED_SPLIT_ID = 'f0dfad2f4431a876'
if not torch.cuda.is_available():
    raise RuntimeError('GPU가 꺼져 있습니다. Kaggle Settings → Accelerator → GPU를 선택하세요.')
print('대회 데이터:', DATA)
print('GPU:', torch.cuda.get_device_name(0), '|', DEVICE)
print('실험 저장 위치:', OUT)
print('공통 Split ID:', EXPECTED_SPLIT_ID)

## 2. 팀 공통 평가 코드 준비 (기존 키트와 동일)

In [ ]:
%%writefile /kaggle/working/teamkit.py
"""Shared, standard-library-only evaluation utilities for the Amazon Top-20 team.

All interactions are targets, regardless of rating. IDs are kept as strings so
CSV round-trips do not change them. The public CSV contract is user_id,item_id.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import math
import os
import sys
import tempfile
import zipfile
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path
from typing import Iterable, Mapping, Sequence


TOP_K = 20
SPLIT_SEED = 42
HOLDOUT_FRACTION = 0.20


@contextmanager
def open_source(path: str | None, competition_zip: str | None, member: str):
    """Open a UTF-8 file, either directly or as a named competition ZIP member."""
    if (path is None) == (competition_zip is None):
        raise ValueError("Provide exactly one of a file path or --competition-zip")
    if competition_zip is not None:
        with zipfile.ZipFile(competition_zip) as archive:
            with archive.open(member) as raw:
                with io.TextIOWrapper(raw, encoding="utf-8-sig", newline="") as stream:
                    yield stream
    else:
        with open(path, "r", encoding="utf-8-sig", newline="") as stream:
            yield stream


def _reader(stream, required: Sequence[str]) -> csv.DictReader:
    reader = csv.DictReader(stream)
    if reader.fieldnames is None or not set(required).issubset(reader.fieldnames):
        raise ValueError(f"Required columns: {', '.join(required)}")
    return reader


def _clean_id(value: str | None, name: str) -> str:
    if value is None or value == "" or value != value.strip() or any(c.isspace() for c in value):
        raise ValueError(f"Invalid {name}: {value!r}")
    return value


def _stable_rank(seed: int, user_id: str, item_id: str) -> bytes:
    payload = f"{seed}\0{user_id}\0{item_id}".encode("utf-8")
    return hashlib.blake2b(payload, digest_size=16).digest()


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _source_sha256(path: str | None, competition_zip: str | None, member: str) -> str:
    digest = hashlib.sha256()
    if competition_zip is not None:
        with zipfile.ZipFile(competition_zip) as archive, archive.open(member) as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
    else:
        with open(path, "rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
    return digest.hexdigest()


def make_split(
    train_path: str | None,
    competition_zip: str | None,
    output_dir: str | Path,
    seed: int = SPLIT_SEED,
    fraction: float = HOLDOUT_FRACTION,
    overwrite: bool = False,
) -> dict:
    """Make a stable, per-user split on unique user-item pairs.

    Hold out ceil(fraction * interactions), at most 20, leaving at least one
    interaction in fit. The hash rank is independent of input row order.
    """
    if not 0 < fraction < 1:
        raise ValueError("fraction must be between 0 and 1")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    names = ("val_train.csv", "val_truth.csv", "val_users.csv", "manifest.json")
    existing = [name for name in names if (output_dir / name).exists()]
    if existing and not overwrite:
        raise FileExistsError(f"Existing split files: {existing}; use --overwrite to replace")

    with open_source(train_path, competition_zip, "train.csv") as stream:
        reader = _reader(stream, ("user_id", "item_id"))
        columns = list(reader.fieldnames or [])
        rows: list[tuple[str, ...]] = []
        user_items: dict[str, list[str]] = defaultdict(list)
        pairs: set[tuple[str, str]] = set()
        for row in reader:
            if None in row:
                raise ValueError("Malformed train.csv row with extra fields")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            pair = (user_id, item_id)
            if pair in pairs:
                raise ValueError(f"Duplicate interaction: {pair}")
            pairs.add(pair)
            user_items[user_id].append(item_id)
            rows.append(tuple(row[column] for column in columns))

    held: set[tuple[str, str]] = set()
    for user_id, items in user_items.items():
        if len(items) < 2:
            continue
        n_held = min(TOP_K, len(items) - 1, max(1, math.ceil(fraction * len(items))))
        chosen = sorted(items, key=lambda item: (_stable_rank(seed, user_id, item), item))[:n_held]
        held.update((user_id, item) for item in chosen)

    fit_path = output_dir / "val_train.csv"
    truth_path = output_dir / "val_truth.csv"
    users_path = output_dir / "val_users.csv"
    user_col, item_col = columns.index("user_id"), columns.index("item_id")
    with fit_path.open("w", encoding="utf-8", newline="") as fit_stream, truth_path.open(
        "w", encoding="utf-8", newline=""
    ) as truth_stream:
        fit_writer, truth_writer = csv.writer(fit_stream), csv.writer(truth_stream)
        fit_writer.writerow(columns)
        truth_writer.writerow(("user_id", "item_id"))
        for row in rows:
            pair = (row[user_col], row[item_col])
            if pair in held:
                truth_writer.writerow(pair)
            else:
                fit_writer.writerow(row)

    with users_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id",))
        for user_id in user_items:
            if len(user_items[user_id]) >= 2:
                writer.writerow((user_id,))

    source_hash = _source_sha256(train_path, competition_zip, "train.csv")
    split_key = f"{source_hash}|blake2b16|{seed}|{fraction}|ceil20keep1"
    manifest = {
        "split_id": hashlib.sha256(split_key.encode()).hexdigest()[:16],
        "algorithm": "per-user blake2b-128 rank over seed, user_id, item_id",
        "seed": seed,
        "holdout_fraction": fraction,
        "holdout_count": "min(20, n-1, max(1, ceil(fraction*n))) for n>=2",
        "source_train_sha256": source_hash,
        "source_rows": len(rows),
        "source_users": len(user_items),
        "fit_rows": len(rows) - len(held),
        "truth_rows": len(held),
        "validation_users": sum(len(items) >= 2 for items in user_items.values()),
        "files_sha256": {name: _sha256(output_dir / name) for name in names[:3]},
    }
    with (output_dir / "manifest.json").open("w", encoding="utf-8") as stream:
        json.dump(manifest, stream, ensure_ascii=False, indent=2)
        stream.write("\n")
    return manifest


def verify_split(output_dir: str | Path) -> dict:
    output_dir = Path(output_dir)
    with (output_dir / "manifest.json").open("r", encoding="utf-8") as stream:
        manifest = json.load(stream)
    for name, expected in manifest["files_sha256"].items():
        actual = _sha256(output_dir / name)
        if actual != expected:
            raise ValueError(f"Split file checksum mismatch: {name}")
    return manifest


def read_users(path: str | Path) -> list[str]:
    users: list[str] = []
    seen: set[str] = set()
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id",)):
            if None in row:
                raise ValueError(f"Malformed user CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            if user_id in seen:
                raise ValueError(f"Duplicate user in {path}: {user_id}")
            seen.add(user_id)
            users.append(user_id)
    return users


def read_interactions(path: str | Path) -> tuple[dict[str, set[str]], Counter[str]]:
    """Return seen items by user and item popularity from a fit CSV."""
    seen: dict[str, set[str]] = defaultdict(set)
    popularity: Counter[str] = Counter()
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id", "item_id")):
            if None in row:
                raise ValueError(f"Malformed interaction CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            if item_id not in seen[user_id]:
                seen[user_id].add(item_id)
                popularity[item_id] += 1
    return dict(seen), popularity


def read_truth(path: str | Path) -> dict[str, set[str]]:
    truth: dict[str, set[str]] = defaultdict(set)
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id", "item_id")):
            if None in row:
                raise ValueError(f"Malformed truth CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            if item_id in truth[user_id]:
                raise ValueError(f"Duplicate truth item for user {user_id}: {item_id}")
            truth[user_id].add(item_id)
    return dict(truth)


def read_predictions(path: str | Path) -> dict[str, list[str]]:
    predictions: dict[str, list[str]] = {}
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["user_id", "item_id"]:
            raise ValueError("Prediction columns must be exactly user_id,item_id")
        for row in reader:
            if None in row:
                raise ValueError("Malformed prediction CSV row")
            user_id = _clean_id(row["user_id"], "user_id")
            if user_id in predictions:
                raise ValueError(f"Duplicate prediction user: {user_id}")
            value = row["item_id"]
            if value is None:
                raise ValueError(f"Missing item list for user {user_id}")
            items = value.split(" ")
            if len(items) != TOP_K or any(not item or item != item.strip() for item in items):
                raise ValueError(f"User {user_id} needs exactly {TOP_K} space-separated item IDs")
            if len(set(items)) != TOP_K:
                raise ValueError(f"Duplicate recommended item for user {user_id}")
            predictions[user_id] = items
    return predictions


def validate_predictions(
    predictions: Mapping[str, Sequence[str]],
    expected_users: Iterable[str],
    seen: Mapping[str, set[str]],
    catalog: set[str],
) -> None:
    expected = set(expected_users)
    actual = set(predictions)
    if actual != expected:
        missing, extra = expected - actual, actual - expected
        raise ValueError(f"User mismatch: {len(missing)} missing, {len(extra)} extra")
    for user_id, items in predictions.items():
        if len(items) != TOP_K or len(set(items)) != TOP_K:
            raise ValueError(f"User {user_id} needs {TOP_K} distinct items")
        unknown = set(items) - catalog
        if unknown:
            raise ValueError(f"Unknown item for user {user_id}: {sorted(unknown)[:3]}")
        repeated = set(items) & seen.get(user_id, set())
        if repeated:
            raise ValueError(f"Seen item recommended to user {user_id}: {sorted(repeated)[:3]}")


def recall_at_20(predictions: Mapping[str, Sequence[str]], truth: Mapping[str, set[str]]) -> dict:
    """Competition-style pooled Recall@20, not mean of user-level recalls."""
    if set(predictions) != set(truth):
        raise ValueError("Prediction and truth user sets differ")
    hits = sum(len(set(items[:TOP_K]) & truth[user_id]) for user_id, items in predictions.items())
    denominator = sum(min(TOP_K, len(items)) for items in truth.values())
    if denominator == 0:
        raise ValueError("No validation truth items")
    return {"recall_at_20": hits / denominator, "hits": hits, "denominator": denominator, "users": len(truth)}


def top20_unseen(
    user_id: str,
    ranked_candidates: Iterable[str],
    seen: Mapping[str, set[str]],
    fallback_items: Iterable[str],
    catalog: set[str],
) -> list[str]:
    """Filter a model ranking; fill short lists with popular unseen items."""
    chosen: list[str] = []
    used = set(seen.get(user_id, set()))
    for source in (ranked_candidates, fallback_items):
        for item_id in source:
            item_id = str(item_id)
            if item_id not in catalog or item_id in used:
                continue
            chosen.append(item_id)
            used.add(item_id)
            if len(chosen) == TOP_K:
                return chosen
    raise ValueError(f"Fewer than {TOP_K} available unseen items for user {user_id}")


def write_predictions(path: str | Path, users: Iterable[str], predictions: Mapping[str, Sequence[str]]) -> None:
    """Write one row per user in the provided order, using single spaces."""
    with open(path, "w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id", "item_id"))
        for user_id in users:
            items = predictions[user_id]
            if len(items) != TOP_K or len(set(items)) != TOP_K:
                raise ValueError(f"User {user_id} needs {TOP_K} distinct items")
            writer.writerow((user_id, " ".join(map(str, items))))


def filter_reviews(
    fit_path: str | Path,
    output_path: str | Path,
    reviews_path: str | None = None,
    competition_zip: str | None = None,
) -> dict:
    """Keep only training-pair reviews to prevent validation target leakage."""
    seen, _ = read_interactions(fit_path)
    allowed = {(user_id, item_id) for user_id, items in seen.items() for item_id in items}
    found: set[tuple[str, str]] = set()
    written = 0
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_path = tempfile.mkstemp(prefix=".reviews-", suffix=".tmp", dir=output_path.parent)
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8", newline="") as output, open_source(
            reviews_path, competition_zip, "train_reviews.jsonl"
        ) as source:
            for line in source:
                review = json.loads(line)
                pair = (str(review["user_id"]), str(review["item_id"]))
                if pair in allowed:
                    output.write(line)
                    found.add(pair)
                    written += 1
        if found != allowed:
            raise ValueError(f"Review coverage mismatch: {len(allowed - found)} fit pairs have no review")
        os.replace(temporary_path, output_path)
    finally:
        if os.path.exists(temporary_path):
            os.unlink(temporary_path)
    return {"written_reviews": written, "distinct_fit_pairs": len(allowed)}


def _print_json(value: dict) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2))


def main(argv: Sequence[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="Shared Amazon Top-20 evaluation tools")
    commands = parser.add_subparsers(dest="command", required=True)

    split = commands.add_parser("split", help="Create deterministic validation CSVs")
    source = split.add_mutually_exclusive_group(required=True)
    source.add_argument("--train", help="Path to extracted train.csv")
    source.add_argument("--competition-zip", help="Path to original competition ZIP")
    split.add_argument("--out", required=True, help="Output directory")
    split.add_argument("--seed", type=int, default=SPLIT_SEED)
    split.add_argument("--fraction", type=float, default=HOLDOUT_FRACTION)
    split.add_argument("--overwrite", action="store_true")

    verify = commands.add_parser("verify-split", help="Check shared split file hashes")
    verify.add_argument("--dir", required=True)

    score = commands.add_parser("score", help="Validate and score validation predictions")
    score.add_argument("--pred", required=True)
    score.add_argument("--fit", required=True)
    score.add_argument("--truth", required=True)

    check = commands.add_parser("check", help="Validate a final test submission")
    check.add_argument("--pred", required=True)
    check.add_argument("--train", required=True)
    check.add_argument("--test", required=True)

    reviews = commands.add_parser("filter-reviews", help="Remove held-out reviews")
    reviews.add_argument("--fit", required=True)
    review_source = reviews.add_mutually_exclusive_group(required=True)
    review_source.add_argument("--reviews", help="Path to extracted train_reviews.jsonl")
    review_source.add_argument("--competition-zip", help="Path to original competition ZIP")
    reviews.add_argument("--out", required=True)

    args = parser.parse_args(argv)
    if args.command == "split":
        _print_json(make_split(args.train, args.competition_zip, args.out, args.seed, args.fraction, args.overwrite))
    elif args.command == "verify-split":
        manifest = verify_split(args.dir)
        _print_json({"split_id": manifest["split_id"], "verified": True})
    elif args.command == "score":
        seen, popularity = read_interactions(args.fit)
        truth = read_truth(args.truth)
        predictions = read_predictions(args.pred)
        catalog = set(popularity) | {item for items in truth.values() for item in items}
        validate_predictions(predictions, truth, seen, catalog)
        result = recall_at_20(predictions, truth)
        result["valid_submission_format"] = True
        _print_json(result)
    elif args.command == "check":
        seen, popularity = read_interactions(args.train)
        users = read_users(args.test)
        predictions = read_predictions(args.pred)
        validate_predictions(predictions, users, seen, set(popularity))
        _print_json({"valid": True, "users": len(users), "items_per_user": TOP_K})
    elif args.command == "filter-reviews":
        _print_json(filter_reviews(args.fit, args.out, args.reviews, args.competition_zip))
    return 0


if __name__ == "__main__":
    try:
        sys.exit(main())
    except (ValueError, FileExistsError, FileNotFoundError, KeyError, zipfile.BadZipFile) as error:
        print(f"ERROR: {error}", file=sys.stderr)
        sys.exit(1)

## 3. LightGCN / NGCF 모델 코드 준비 (이전 실험과 동일)

In [ ]:
%%writefile /kaggle/working/graph_models.py
"""Single-GPU LightGCN / NGCF experiments for the shared Amazon Top-20 split.

Example:
    python graph_models.py --model lightgcn \
      --fit ../amazon_topk_teamkit/validation/val_train.csv \
      --users ../amazon_topk_teamkit/validation/val_users.csv \
      --truth ../amazon_topk_teamkit/validation/val_truth.csv \
      --out lightgcn_val.csv

No metadata or reviews are read. Train interactions are the only graph edges.
"""

from __future__ import annotations

import argparse
import copy
import csv
import json
import math
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Sequence

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ModuleNotFoundError:
    torch = None
    nn = None
    F = None


TOP_K = 20
_BaseModel = nn.Module if nn is not None else object


def _id_key(value: str):
    return (0, int(value)) if value.isdecimal() else (1, value)


def _csv_rows(path: str | Path, required: tuple[str, ...]):
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames is None or not set(required).issubset(reader.fieldnames):
            raise ValueError(f"{path} must contain {', '.join(required)}")
        for row in reader:
            if None in row:
                raise ValueError(f"Malformed CSV row in {path}")
            yield row


def _identifier(value: str | None, column: str) -> str:
    if not value or value != value.strip() or any(char.isspace() for char in value):
        raise ValueError(f"Invalid {column}: {value!r}")
    return value


@dataclass
class InteractionData:
    user_ids: list[str]
    item_ids: list[str]
    user_index: dict[str, int]
    item_index: dict[str, int]
    positive_users: list[int]
    positive_items: list[int]
    seen_items: dict[int, set[int]]
    target_users: list[str]

    @property
    def num_users(self) -> int:
        return len(self.user_ids)

    @property
    def num_items(self) -> int:
        return len(self.item_ids)

    @property
    def num_edges(self) -> int:
        return len(self.positive_users)


def load_data(fit_path: str | Path, users_path: str | Path) -> InteractionData:
    """Index unique positive edges; do not filter interactions by rating."""
    edges: set[tuple[str, str]] = set()
    for row in _csv_rows(fit_path, ("user_id", "item_id")):
        edges.add((_identifier(row["user_id"], "user_id"), _identifier(row["item_id"], "item_id")))
    if not edges:
        raise ValueError("Empty fit interactions")

    target_users: list[str] = []
    target_set: set[str] = set()
    for row in _csv_rows(users_path, ("user_id",)):
        user_id = _identifier(row["user_id"], "user_id")
        if user_id in target_set:
            raise ValueError(f"Duplicate target user: {user_id}")
        target_set.add(user_id)
        target_users.append(user_id)

    train_users = {user for user, _ in edges}
    unseen_users = target_set - train_users
    if unseen_users:
        raise ValueError(f"{len(unseen_users)} target users have no training interaction")
    user_ids = sorted(train_users, key=_id_key)
    item_ids = sorted({item for _, item in edges}, key=_id_key)
    if len(item_ids) <= TOP_K:
        raise ValueError("Need more than 20 known items to recommend 20 unseen items")
    user_index = {value: index for index, value in enumerate(user_ids)}
    item_index = {value: index for index, value in enumerate(item_ids)}
    positive_users: list[int] = []
    positive_items: list[int] = []
    seen_items: dict[int, set[int]] = defaultdict(set)
    for user, item in sorted(edges, key=lambda pair: (_id_key(pair[0]), _id_key(pair[1]))):
        u, i = user_index[user], item_index[item]
        positive_users.append(u)
        positive_items.append(i)
        seen_items[u].add(i)
    for user in target_users:
        if len(seen_items[user_index[user]]) + TOP_K > len(item_ids):
            raise ValueError(f"User {user} has fewer than 20 unseen items")
    return InteractionData(
        user_ids, item_ids, user_index, item_index, positive_users,
        positive_items, dict(seen_items), target_users,
    )


def load_truth(path: str | Path, target_users: Sequence[str]) -> dict[str, set[str]]:
    truth: dict[str, set[str]] = defaultdict(set)
    for row in _csv_rows(path, ("user_id", "item_id")):
        user, item = _identifier(row["user_id"], "user_id"), _identifier(row["item_id"], "item_id")
        if item in truth[user]:
            raise ValueError(f"Duplicate truth pair: {user}, {item}")
        truth[user].add(item)
    if set(truth) != set(target_users):
        raise ValueError("Truth and target user sets differ")
    return dict(truth)


def recall_at_20(predictions: dict[str, list[str]], truth: dict[str, set[str]]) -> dict:
    """Pooled competition Recall@20: total hits / sum(min(20, |GT_u|))."""
    if set(predictions) != set(truth):
        raise ValueError("Prediction and truth user sets differ")
    hits = sum(len(set(items[:TOP_K]) & truth[user]) for user, items in predictions.items())
    denominator = sum(min(TOP_K, len(items)) for items in truth.values())
    if not denominator:
        raise ValueError("No truth items")
    return {"recall_at_20": hits / denominator, "hits": hits, "denominator": denominator, "users": len(truth)}


def normalized_graph(data: InteractionData, device):
    """D^-1/2 A D^-1/2 for the undirected user-item graph."""
    if torch is None:
        raise RuntimeError("PyTorch is required to build the graph")
    u = torch.tensor(data.positive_users, dtype=torch.long)
    i = torch.tensor(data.positive_items, dtype=torch.long) + data.num_users
    rows = torch.cat((u, i))
    cols = torch.cat((i, u))
    degree = torch.bincount(rows, minlength=data.num_users + data.num_items).float()
    values = (degree[rows] * degree[cols]).rsqrt()
    indices = torch.stack((rows, cols)).to(device)
    return torch.sparse_coo_tensor(
        indices, values.to(device),
        (data.num_users + data.num_items, data.num_users + data.num_items),
        device=device,
    ).coalesce()


def sparse_edge_dropout(graph, probability: float):
    if probability <= 0:
        return graph
    if not 0 <= probability < 1:
        raise ValueError("Edge dropout must be in [0, 1)")
    keep = torch.rand(graph._nnz(), device=graph.device) >= probability
    return torch.sparse_coo_tensor(
        graph.indices()[:, keep], graph.values()[keep] / (1 - probability),
        graph.shape, device=graph.device,
    ).coalesce()


class LightGCN(_BaseModel):
    def __init__(self, n_users: int, n_items: int, dim: int, layers: int):
        if nn is None:
            raise RuntimeError("PyTorch is required to train LightGCN")
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.item_embedding = nn.Embedding(n_items, dim)
        self.layers = layers
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def forward(self, graph):
        current = torch.cat((self.user_embedding.weight, self.item_embedding.weight), dim=0)
        layer_outputs = [current]
        for _ in range(self.layers):
            current = torch.sparse.mm(graph, current)
            layer_outputs.append(current)
        final = torch.stack(layer_outputs, dim=0).mean(dim=0)
        return final[: self.user_embedding.num_embeddings], final[self.user_embedding.num_embeddings :]


class NGCF(_BaseModel):
    def __init__(self, n_users: int, n_items: int, dim: int, layers: int, message_dropout: float):
        if nn is None:
            raise RuntimeError("PyTorch is required to train NGCF")
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.item_embedding = nn.Embedding(n_items, dim)
        self.sum_linears = nn.ModuleList(nn.Linear(dim, dim) for _ in range(layers))
        self.bi_linears = nn.ModuleList(nn.Linear(dim, dim) for _ in range(layers))
        self.message_dropout = message_dropout
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)
        for layer in (*self.sum_linears, *self.bi_linears):
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, graph):
        current = torch.cat((self.user_embedding.weight, self.item_embedding.weight), dim=0)
        layer_outputs = [current]
        for sum_linear, bi_linear in zip(self.sum_linears, self.bi_linears):
            neighbors = torch.sparse.mm(graph, current)
            current = F.leaky_relu(sum_linear(current + neighbors) + bi_linear(current * neighbors), negative_slope=0.2)
            current = F.dropout(current, p=self.message_dropout, training=self.training)
            current = F.normalize(current, p=2, dim=1)
            layer_outputs.append(current)
        final = torch.cat(layer_outputs, dim=1)
        return final[: self.user_embedding.num_embeddings], final[self.user_embedding.num_embeddings :]


def sample_triples(data: InteractionData, count: int, rng: random.Random, device):
    """Sample one uniformly random unseen item per sampled positive edge."""
    if count <= 0:
        raise ValueError("samples-per-epoch must be positive")
    edge_indices = [rng.randrange(data.num_edges) for _ in range(count)]
    users = [data.positive_users[index] for index in edge_indices]
    positive = [data.positive_items[index] for index in edge_indices]
    negative = [rng.randrange(data.num_items) for _ in range(count)]
    for position, (user, item) in enumerate(zip(users, negative)):
        if item in data.seen_items[user]:
            while True:
                candidate = rng.randrange(data.num_items)
                if candidate not in data.seen_items[user]:
                    negative[position] = candidate
                    break
    return (
        torch.tensor(users, dtype=torch.long, device=device),
        torch.tensor(positive, dtype=torch.long, device=device),
        torch.tensor(negative, dtype=torch.long, device=device),
    )


@torch.no_grad() if torch is not None else (lambda function: function)
def predict_top20(model, graph, data: InteractionData, user_batch_size: int) -> dict[str, list[str]]:
    """Score all known items in bounded user batches and mask fit interactions."""
    if user_batch_size <= 0:
        raise ValueError("eval-user-batch-size must be positive")
    model.eval()
    user_vectors, item_vectors = model(graph)
    item_matrix = item_vectors.T.contiguous()
    predictions: dict[str, list[str]] = {}
    for start in range(0, len(data.target_users), user_batch_size):
        batch = data.target_users[start : start + user_batch_size]
        user_indices = torch.tensor([data.user_index[user] for user in batch], dtype=torch.long, device=graph.device)
        scores = user_vectors[user_indices] @ item_matrix
        masked_rows: list[int] = []
        masked_columns: list[int] = []
        for local_row, user_index in enumerate(user_indices.tolist()):
            for item_index in data.seen_items[user_index]:
                masked_rows.append(local_row)
                masked_columns.append(item_index)
        if masked_rows:
            row_index = torch.tensor(masked_rows, dtype=torch.long, device=graph.device)
            column_index = torch.tensor(masked_columns, dtype=torch.long, device=graph.device)
            scores[row_index, column_index] = -float("inf")
        top = torch.topk(scores, k=TOP_K, dim=1).indices.cpu().tolist()
        for user_id, item_indices in zip(batch, top):
            predictions[user_id] = [data.item_ids[index] for index in item_indices]
    return predictions


def write_predictions(path: str | Path, users: Sequence[str], predictions: dict[str, list[str]]) -> None:
    with open(path, "w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id", "item_id"))
        for user in users:
            writer.writerow((user, " ".join(predictions[user])))


def _device(name: str):
    if torch is None:
        raise RuntimeError("PyTorch is not installed. Use Kaggle GPU Notebook or install torch.")
    if name == "auto":
        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    if name.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable")
    return torch.device(name)


def _training_loss(model, user_vectors, item_vectors, sampled, regularization: float):
    users, positives, negatives = sampled
    positive_scores = (user_vectors[users] * item_vectors[positives]).sum(dim=1)
    negative_scores = (user_vectors[users] * item_vectors[negatives]).sum(dim=1)
    bpr = F.softplus(negative_scores - positive_scores).mean()
    raw_u = model.user_embedding(users)
    raw_p = model.item_embedding(positives)
    raw_n = model.item_embedding(negatives)
    reg = (raw_u.square().sum(dim=1) + raw_p.square().sum(dim=1) + raw_n.square().sum(dim=1)).mean() / 2
    return bpr + regularization * reg, bpr, reg


def run(args: argparse.Namespace) -> dict:
    device = _device(args.device)
    if args.layers < 1 or args.embedding_dim < 1 or args.epochs < 1:
        raise ValueError("layers, embedding-dim and epochs must be positive")
    if not 0 <= args.edge_dropout < 1 or not 0 <= args.message_dropout < 1:
        raise ValueError("Dropout probabilities must be in [0, 1)")
    if args.reg < 0 or args.lr <= 0:
        raise ValueError("reg must be nonnegative and lr positive")
    torch.manual_seed(args.seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(args.seed)
    rng = random.Random(args.seed)
    data = load_data(args.fit, args.users)
    truth = load_truth(args.truth, data.target_users) if args.truth else None
    graph = normalized_graph(data, device)
    if args.model == "lightgcn":
        model = LightGCN(data.num_users, data.num_items, args.embedding_dim, args.layers)
    else:
        model = NGCF(data.num_users, data.num_items, args.embedding_dim, args.layers, args.message_dropout)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    print(json.dumps({
        "model": args.model, "device": str(device), "users": data.num_users,
        "items": data.num_items, "fit_edges": data.num_edges,
        "embedding_dim": args.embedding_dim, "layers": args.layers,
        "samples_per_epoch": args.samples_per_epoch,
    }), flush=True)

    best_score = -math.inf
    best_state = None
    best_epoch = None
    stale_evaluations = 0
    last_loss = None
    trained_epochs = 0
    for epoch in range(1, args.epochs + 1):
        trained_epochs = epoch
        start = time.monotonic()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        sampled = sample_triples(data, args.samples_per_epoch, rng, device)
        epoch_graph = sparse_edge_dropout(graph, args.edge_dropout) if args.model == "ngcf" else graph
        user_vectors, item_vectors = model(epoch_graph)
        loss, bpr, reg = _training_loss(model, user_vectors, item_vectors, sampled, args.reg)
        loss.backward()
        optimizer.step()
        last_loss = float(loss.detach().cpu())
        record = {
            "epoch": epoch, "loss": round(last_loss, 6),
            "bpr": round(float(bpr.detach().cpu()), 6),
            "regularization": round(float(reg.detach().cpu()), 6),
            "seconds": round(time.monotonic() - start, 2),
        }
        del sampled, user_vectors, item_vectors, loss, bpr, reg
        should_evaluate = truth is not None and (epoch == 1 or epoch % args.eval_every == 0 or epoch == args.epochs)
        if should_evaluate:
            predictions = predict_top20(model, graph, data, args.eval_user_batch_size)
            metric = recall_at_20(predictions, truth)
            record.update(metric)
            if metric["recall_at_20"] > best_score:
                best_score = metric["recall_at_20"]
                best_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                stale_evaluations = 0
                record["best"] = True
            else:
                stale_evaluations += 1
                record["best"] = False
        print(json.dumps(record), flush=True)
        if should_evaluate and args.patience > 0 and stale_evaluations >= args.patience:
            print(json.dumps({"early_stop_epoch": epoch}), flush=True)
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    predictions = predict_top20(model, graph, data, args.eval_user_batch_size)
    write_predictions(args.out, data.target_users, predictions)
    result = {
        "model": args.model, "device": str(device), "fit_edges": data.num_edges,
        "users": len(data.target_users), "output": str(args.out), "last_loss": last_loss,
        "trained_epochs": trained_epochs, "best_epoch": best_epoch,
        "epochs_requested": args.epochs, "seed": args.seed,
        "embedding_dim": args.embedding_dim, "layers": args.layers,
        "samples_per_epoch": args.samples_per_epoch,
        "learning_rate": args.lr, "regularization": args.reg,
        "edge_dropout": args.edge_dropout if args.model == "ngcf" else 0.0,
        "message_dropout": args.message_dropout if args.model == "ngcf" else 0.0,
    }
    if truth is not None:
        result.update(recall_at_20(predictions, truth))
    print(json.dumps({"final": result}), flush=True)
    return result


def main(argv: Sequence[str] | None = None) -> dict:
    parser = argparse.ArgumentParser(description="LightGCN / NGCF on the shared Amazon Top-20 split")
    parser.add_argument("--model", choices=("lightgcn", "ngcf"), required=True)
    parser.add_argument("--fit", required=True, help="val_train.csv or full train.csv")
    parser.add_argument("--users", required=True, help="val_users.csv or test.csv")
    parser.add_argument("--truth", help="val_truth.csv; omit for final test predictions")
    parser.add_argument("--out", required=True, help="Prediction CSV")
    parser.add_argument("--metrics-out", help="Optional JSON file for run settings and results")
    parser.add_argument("--epochs", type=int, default=40)
    parser.add_argument("--embedding-dim", type=int, default=64)
    parser.add_argument("--layers", type=int, default=3)
    parser.add_argument("--samples-per-epoch", type=int, default=100000)
    parser.add_argument("--eval-user-batch-size", type=int, default=256)
    parser.add_argument("--eval-every", type=int, default=5)
    parser.add_argument("--patience", type=int, default=4, help="Validation evaluations without improvement; 0 disables")
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--reg", type=float, default=0.0001)
    parser.add_argument("--edge-dropout", type=float, default=0.0, help="Optional NGCF edge dropout")
    parser.add_argument("--message-dropout", type=float, default=0.1, help="NGCF only")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default="auto", help="auto, cpu, cuda:0")
    args = parser.parse_args(argv)
    if args.eval_every < 1 or args.patience < 0:
        parser.error("eval-every must be positive and patience nonnegative")
    result = run(args)
    if args.metrics_out:
        Path(args.metrics_out).write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


if __name__ == "__main__":
    main()

## 4. 기존 팀 공통 ZIP을 읽고 무결성 확인

In [ ]:
import shutil
import zipfile
sys.path.insert(0, str(WORK))
from teamkit import verify_split, read_truth

required = ('val_train.csv', 'val_truth.csv', 'val_users.csv', 'manifest.json')
EXPECTED_SPLIT_ID = 'f0dfad2f4431a876'

# Kaggle에 등록한 ZIP은 대개 자동 압축 해제되어 /kaggle/input 안에 보입니다.
source_directory = None
source_zip = None
for base in (Path('/kaggle/input'), WORK):
    for manifest_path in base.rglob('manifest.json'):
        folder = manifest_path.parent
        if folder != VALID and all((folder / name).is_file() for name in required):
            # 여러 폴더 중 정확한 팀 공통 분할만 선택합니다.
            try:
                meta = json.loads(manifest_path.read_text(encoding='utf-8'))
            except (ValueError, OSError):
                continue
            if meta.get('split_id') == EXPECTED_SPLIT_ID:
                source_directory = folder
                break
    if source_directory is not None:
        break

if source_directory is None:
    for base in (Path('/kaggle/input'), WORK):
        candidates = list(base.rglob('amazon_topk_teamkit_20261110.zip'))
        if candidates:
            source_zip = candidates[0]
            break

if source_directory is None and source_zip is None:
    raise FileNotFoundError(
        '팀 공통 ZIP을 Input에 추가하지 않았습니다. '
        '오른쪽 Upload에서 amazon_topk_teamkit_20261110.zip을 '
        '비공개 데이터셋으로 올린 후 이 셀을 다시 실행하세요.'
    )

VALID.mkdir(parents=True, exist_ok=True)
if source_directory is not None:
    for name in required:
        shutil.copyfile(source_directory / name, VALID / name)
    print('Shared validation loaded from:', source_directory)
else:
    with zipfile.ZipFile(source_zip) as archive:
        for name in required:
            matches = [p for p in archive.namelist() if p.endswith('/validation/' + name)]
            if len(matches) != 1:
                raise RuntimeError(f'{name}: ZIP에 파일을 정확히 하나 찾지 못했습니다. {matches}')
            with archive.open(matches[0]) as from_zip, (VALID / name).open('wb') as out:
                shutil.copyfileobj(from_zip, out)
    print('Shared validation extracted from:', source_zip)

manifest = verify_split(VALID)
assert manifest['split_id'] == EXPECTED_SPLIT_ID, '다른 팀 분할입니다.'
train_hasher = hashlib.sha256()
with TRAIN.open('rb') as f:
    for part in iter(lambda: f.read(1024 * 1024), b''):
        train_hasher.update(part)
train_hash = train_hasher.hexdigest()
assert train_hash == manifest['source_train_sha256'], '대회 train.csv가 팀 공통 분할 원본과 다릅니다.'
truth = read_truth(VALID / 'val_truth.csv')
assert len(truth) == manifest['validation_users']
print('Validation matched team split:', len(truth), 'users,', manifest['truth_rows'], 'held-out interactions')
print('Split ID:', manifest['split_id'])
print('Training interactions:', manifest['fit_rows'])

## 5. Ablation 실험 계획 선택

- **MAX (기본):** 모두 실행. 기본 설정, 레이어 1/2/3, LR 0.0015/0.002/0.003, 차원 32/64/128, 정규화, 학습 길이, 샘플 수 및 NGCF 구조/드롭아웃.
- **BALANCED:** 영향이 클 것으로 예상되는 실험만 우선 실행.
- **QUICK:** 정상 실행 확인과 핵심 조합 위주.
- 현재 최고 기준점은 *이전 실험* LightGCN 3-layer, dim64, LR0.002, 700 Epoch의 **0.034391**이지만, **이 수치를 이번 실험의 측정값으로 속여 넣거나 제출 모델 선택에 사용하지 않습니다.** 첫 실험이 동일 설정을 재현합니다.
- `MAKE_SUBMISSION=True`이면 이번 실험에서 **실제로 확인한 1등 설정**을 전체 train에 다시 학습해 `/kaggle/working/submission.csv` 하나를 만듭니다. 노트북이 끝나도 Kaggle 대회 제출 버튼은 직접 눌러야 합니다.

In [ ]:
RUN_PROFILE = 'MAX'  # 'MAX' | 'BALANCED' | 'QUICK'
MAKE_SUBMISSION = True  # 제출 파일 자동 생성. Kaggle 서버로 실제 제출하지는 않음.
SKIP_COMPLETED = True  # 같은 노트북 실행 환경에서 이미 성공한 동일 실험이면 건너뜀

# 각각의 실험은 전체 Validation을 사용하므로 빠른 확인용 사용자 샘플링을 하지 않습니다.
# 모든 모델은 동일 평가를 위해 매 25 Epoch마다 Recall@20을 계산합니다.

def conf(name, model='lightgcn', layers=3, dim=64, lr=0.002, reg=0.0001,
         epochs=700, samples=100000, message_dropout=0.0, edge_dropout=0.0):
    return dict(name=name, model=model, layers=layers, embedding_dim=dim,
                lr=lr, reg=reg, epochs=epochs, samples_per_epoch=samples,
                message_dropout=message_dropout, edge_dropout=edge_dropout)

ALL_EXPERIMENTS = [
    # 이전 최고 점수 설정 재현: 이번 실행에서 검증 후에만 최종 제출 후보로 사용
    conf('L00_reference_lr002_3L_64d'),
    # 1. 그래프 레이어 ablation
    conf('L01_layers2', layers=2),
    conf('L02_layers1', layers=1),
    # 2. 학습률 ablation
    conf('L03_lr0015', lr=0.0015),
    conf('L04_lr003', lr=0.003),
    # 3. 임베딩 차원 ablation
    conf('L05_dim32', dim=32),
    conf('L06_dim128', dim=128),
    # 4. BPR 임베딩 정규화 ablation
    conf('L07_reg00001', reg=0.00001),
    conf('L08_reg0005', reg=0.0005),
    conf('L09_reg001', reg=0.001),
    # 5. 학습 길이 / 샘플 수 ablation
    conf('L10_epochs1100', epochs=1100),
    conf('L11_samples200k', samples=200000),
    # 6. 효과가 있는 조건들의 조합 (단일 변수 실험은 위 항목)
    conf('L12_layers2_lr003', layers=2, lr=0.003, epochs=800),
    conf('L13_layers2_dim128', layers=2, dim=128, epochs=800),
    conf('L14_dim128_lr003', dim=128, lr=0.003, epochs=800),
    # 7. NGCF는 이미 2L/800e로 0.01914를 기록한 적이 있음. 추가 비교
    conf('N01_2L_lr002', model='ngcf', layers=2, lr=0.002, epochs=1000,
         message_dropout=0.1),
    conf('N02_1L_lr001', model='ngcf', layers=1, lr=0.001, epochs=1000,
         message_dropout=0.1),
    conf('N03_2L_dropout02', model='ngcf', layers=2, lr=0.001, epochs=1000,
         message_dropout=0.2),
    conf('N04_2L_reg001', model='ngcf', layers=2, lr=0.001, epochs=1000,
         reg=0.001, message_dropout=0.1),
]

PROFILES = {
    'QUICK': {'L00_reference_lr002_3L_64d', 'L01_layers2', 'L04_lr003',
              'L06_dim128', 'N01_2L_lr002'},
    'BALANCED': {'L00_reference_lr002_3L_64d', 'L01_layers2', 'L02_layers1',
                 'L04_lr003', 'L05_dim32', 'L06_dim128', 'L07_reg00001',
                 'L09_reg001', 'L10_epochs1100', 'L12_layers2_lr003',
                 'L13_layers2_dim128', 'N01_2L_lr002', 'N03_2L_dropout02'},
    'MAX': {c['name'] for c in ALL_EXPERIMENTS},
}
assert RUN_PROFILE in PROFILES
EXPERIMENTS = [c for c in ALL_EXPERIMENTS if c['name'] in PROFILES[RUN_PROFILE]]
assert len(EXPERIMENTS) == len({c['name'] for c in EXPERIMENTS})
print(f'PROFILE={RUN_PROFILE}, 총 {len(EXPERIMENTS)}개 실험, 총 계획 Epoch={sum(c["epochs"] for c in EXPERIMENTS)}')
for c in EXPERIMENTS:
    print(f'{c["name"]:28s} {c["model"]:8s} L{c["layers"]} dim{c["embedding_dim"]} '
          f'lr={c["lr"]} reg={c["reg"]} epochs={c["epochs"]} '
          f'dropout={c["message_dropout"]} sample={c["samples_per_epoch"]}')

## 6. 실험 실행 · 실시간 요약 로그 · 결과 자동 보존

모든 Epoch의 원본 로그는 `ablation_top20/<이름>.log`에 저장하고, 화면에는 **Validation 평가가 수행된 Epoch만** 출력합니다. 에러가 나면 다음 실험을 계속하되 `failed_experiments.json`에 기록합니다.

검증 점수와 모델 설정은 **각 실험이 끝날 때마다 즉시 CSV/JSON에 저장**됩니다. 중간에 실행을 멈춰도 완료된 결과를 확인할 수 있습니다.

In [ ]:
from teamkit import read_interactions, read_predictions, validate_predictions, recall_at_20, read_truth

FIT_PATH = VALID / 'val_train.csv'
USERS_PATH = VALID / 'val_users.csv'
TRUTH_PATH = VALID / 'val_truth.csv'
seen, pop = read_interactions(FIT_PATH)
truth = read_truth(TRUTH_PATH)
catalog = set(pop) | {item for items in truth.values() for item in items}
MODEL_SHA256 = hashlib.sha256((WORK/'graph_models.py').read_bytes()).hexdigest()

def config_stamp(config):
    return {'split_id': EXPECTED_SPLIT_ID, 'graph_sha256': MODEL_SHA256, 'config': config}

def result_paths(name):
    return {
        'pred': OUT / f'{name}_val.csv',
        'metrics': OUT / f'{name}.metrics.json',
        'log': OUT / f'{name}.log',
        'config': OUT / f'{name}.config.json',
    }

def _build_args(config, *, fit, users, out, metrics, truth_file=None, epochs=None):
    args = [
        sys.executable, str(WORK/'graph_models.py'),
        '--model', config['model'],
        '--fit', str(fit), '--users', str(users),
        '--out', str(out), '--metrics-out', str(metrics),
        '--epochs', str(config['epochs'] if epochs is None else epochs),
        '--layers', str(config['layers']),
        '--embedding-dim', str(config['embedding_dim']),
        '--lr', str(config['lr']), '--reg', str(config['reg']),
        '--samples-per-epoch', str(config['samples_per_epoch']),
        '--message-dropout', str(config['message_dropout']),
        '--edge-dropout', str(config['edge_dropout']),
        '--eval-user-batch-size', '256', '--eval-every', '25',
        '--patience', '0', '--seed', str(SEED), '--device', DEVICE,
    ]
    if truth_file is not None:
        args.extend(['--truth', str(truth_file)])
    return args

def run_process(command, log_file, label):
    print(f'\n===== START {label} =====', flush=True)
    with log_file.open('w', encoding='utf-8') as f:
        with subprocess.Popen(command, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            tail = []
            for line in proc.stdout:
                f.write(line)
                if len(tail) >= 18:
                    tail.pop(0)
                tail.append(line.rstrip())
                try:
                    data = json.loads(line)
                except ValueError:
                    if 'Traceback' in line or 'Error' in line:
                        print(line.rstrip(), flush=True)
                    continue
                if 'epoch' in data and 'recall_at_20' in data:
                    print(f'  epoch={data["epoch"]:4d}  '
                          f'Recall@20={data["recall_at_20"]:.6f}  '
                          f'hits={data["hits"]}  best={data.get("best", False)}', flush=True)
                elif 'final' in data:
                    print('  final:', json.dumps(data['final'], ensure_ascii=False), flush=True)
            returncode = proc.wait()
    if returncode:
        raise RuntimeError(f'{label} failed (exit={returncode}). Last logs:\n' + '\n'.join(tail))
    print(f'===== DONE {label} =====', flush=True)

def validated_row(config):
    p = result_paths(config['name'])
    if not all(p[k].is_file() for k in ('pred','metrics','config')):
        return None
    if json.loads(p['config'].read_text(encoding='utf-8')) != config_stamp(config):
        return None
    preds = read_predictions(p['pred'])
    validate_predictions(preds, truth, seen, catalog)
    score = recall_at_20(preds, truth)
    details = json.loads(p['metrics'].read_text(encoding='utf-8'))
    if abs(float(details['recall_at_20']) - score['recall_at_20']) > 1e-12:
        raise AssertionError(f'{config["name"]}: 모델과 공통 평가 점수가 다릅니다.')
    return {
        'experiment': config['name'], 'model': config['model'],
        'layers': config['layers'], 'embedding_dim': config['embedding_dim'],
        'lr': config['lr'], 'reg': config['reg'], 'message_dropout': config['message_dropout'],
        'edge_dropout': config['edge_dropout'], 'samples_per_epoch': config['samples_per_epoch'],
        'epochs_requested': config['epochs'], 'best_epoch': details['best_epoch'],
        'trained_epochs': details['trained_epochs'], 'recall_at_20': score['recall_at_20'],
        'hits': score['hits'], 'denominator': score['denominator'],
        'split_id': EXPECTED_SPLIT_ID,
    }

def collect_rows():
    rows = []
    for c in EXPERIMENTS:
        row = validated_row(c)
        if row is not None:
            rows.append(row)
    return sorted(rows, key=lambda x:(-x['recall_at_20'], x['experiment']))

def save_leaderboard(rows):
    if not rows:
        return
    with (OUT/'ablation_results.csv').open('w', encoding='utf-8', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader(); w.writerows(rows)
    (OUT/'ablation_results.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')

print('공통 Validation 준비 완료:', len(truth), 'users,', sum(map(len, truth.values())), 'truth items')

## 7. 전체 Ablation 실행 (한 셀에서 순차적으로 수행)

In [ ]:
failures = {}
for number, c in enumerate(EXPERIMENTS, 1):
    p = result_paths(c['name'])
    print(f'\n[{number}/{len(EXPERIMENTS)}] {c["name"]}', flush=True)
    if SKIP_COMPLETED:
        try:
            saved = validated_row(c)
        except Exception as exc:
            print('기존 파일 검증에 실패하여 재학습:', str(exc)[:250])
            saved = None
        if saved is not None:
            print(f'  SKIP: 이미 완료된 동일한 설정, Recall@20={saved["recall_at_20"]:.6f}')
            save_leaderboard(collect_rows())
            continue
    for path in (p['pred'], p['metrics'], p['config']):
        path.unlink(missing_ok=True)
    command = _build_args(c, fit=FIT_PATH, users=USERS_PATH, out=p['pred'],
                          metrics=p['metrics'], truth_file=TRUTH_PATH)
    try:
        run_process(command, p['log'], c['name'])
        # 완성된 모델 결과만 재현 가능한 것으로 등록
        p['config'].write_text(json.dumps(config_stamp(c), ensure_ascii=False, indent=2), encoding='utf-8')
        row = validated_row(c)
        if row is None:
            raise RuntimeError('학습 완료 후 예측 또는 평가 파일이 없습니다.')
        print(f'  VALIDATED: Recall@20={row["recall_at_20"]:.6f}; best_epoch={row["best_epoch"]}')
        save_leaderboard(collect_rows())
    except Exception as exc:
        p['config'].unlink(missing_ok=True)
        failures[c['name']] = str(exc)
        (OUT/'failed_experiments.json').write_text(json.dumps(failures, ensure_ascii=False, indent=2), encoding='utf-8')
        print('  FAILED:', str(exc)[:1200], flush=True)
        print('  다음 실험으로 진행합니다. (상세 로그:', p['log'], ')', flush=True)

all_rows = collect_rows()
save_leaderboard(all_rows)
print('\n완료된 실험:', len(all_rows), '/', len(EXPERIMENTS))
print('실패:', len(failures))
if not all_rows:
    raise RuntimeError('성공한 실험이 하나도 없습니다. 로그를 확인하세요.')
print('현재 1등:', all_rows[0]['experiment'], all_rows[0]['recall_at_20'])

## 8. 최종 점수표와 Ablation 분석표 확인

In [ ]:
import pandas as pd
rows = collect_rows()
if not rows:
    raise RuntimeError('실험 결과가 없어요. 위 Ablation 셀을 먼저 실행하세요.')
comparison = pd.DataFrame(rows).sort_values('recall_at_20', ascending=False).reset_index(drop=True)
display(comparison[['experiment','model','layers','embedding_dim','lr','reg',
                    'epochs_requested','best_epoch','recall_at_20','hits']])
print('\n기존 참고 기록: LightGCN / 3L / 64d / LR0.002 / 700e: Recall@20 ≈ 0.034391')
print('이전 기록은 이번에 실제 학습한 모델이 아니면 1등 모델 선정에서 제외합니다.')
print('현재 최고:', comparison.iloc[0]['experiment'], '\nRecall@20:', comparison.iloc[0]['recall_at_20'])
print('점수표:', OUT/'ablation_results.csv')

## 9. 1등 모델 하나로 최종 제출 CSV 생성하기

**이 셀은 `MAKE_SUBMISSION=True`일 때 실행됩니다.** Validation에서 실제로 검증된 최고 모델의 설정과 best_epoch를 가져와, **원본 전체 train.csv로 새로 학습**한 후 대회 `test.csv` 전원에 대해 Top-20을 예측합니다.

- Validation의 정답이나 검증 예측을 최종 학습/추천 입력으로 사용하지 않습니다.
- 최종 학습은 1등 설정을 그대로 쓰되, `epochs=best_epoch`로 새로 처음부터 학습합니다. (체크포인트를 이어받는 방식 아님)
- 팀 키트 `check`로 모든 사용자·Top20·중복·기존 상호작용 제외를 확인합니다.
- **파일은 하나만** `/kaggle/working/submission.csv`에 만들고, Kaggle 페이지에서 사용자가 직접 제출합니다.

In [ ]:
if not MAKE_SUBMISSION:
    print('최종 제출 파일 생성을 건너뜁니다. MAKE_SUBMISSION=False')
else:
    measured_rows = collect_rows()
    if not measured_rows:
        raise RuntimeError('Validation 완료된 모델이 없습니다. 먼저 실험하세요.')
    selected = measured_rows[0]
    best_config = next(x for x in EXPERIMENTS if x['name'] == selected['experiment'])
    selected_epoch = int(selected['best_epoch'])
    if selected_epoch <= 0:
        raise RuntimeError('best_epoch 값이 잘못되었습니다.')
    print('1등 설정:', json.dumps(best_config, indent=2, ensure_ascii=False))
    print('Validation Recall@20:', selected['recall_at_20'], '; 최적 Epoch:', selected_epoch)

    submission = WORK/'submission.csv'
    submission_meta = OUT/'submission.metrics.json'
    final_log = OUT/'final_train.log'
    submission.unlink(missing_ok=True)
    cmd = _build_args(best_config, fit=TRAIN, users=TEST, out=submission,
                      metrics=submission_meta, truth_file=None, epochs=selected_epoch)
    run_process(cmd, final_log, f'FINAL {best_config["name"]}')
    if not submission.is_file():
        raise FileNotFoundError('submission.csv 생성 실패')
    subprocess.run([sys.executable, str(WORK/'teamkit.py'), 'check',
                    '--pred', str(submission), '--train', str(TRAIN),
                    '--test', str(TEST)], check=True)
    from teamkit import read_predictions, read_users
    test_users = read_users(TEST)
    predictions = read_predictions(submission)
    assert len(predictions) == len(test_users)
    (OUT/'selected_submission_config.json').write_text(json.dumps({
        'selected_experiment': selected['experiment'],
        'validation_recall_at_20': selected['recall_at_20'],
        'validation_best_epoch': selected_epoch,
        'split_id': EXPECTED_SPLIT_ID,
        'full_training_config': best_config,
        'submission_path': str(submission),
        'test_user_count': len(test_users),
    }, ensure_ascii=False, indent=2), encoding='utf-8')
    print('\n✅ 제출 파일 검사 통과:', submission)
    print('사용자 수:', len(test_users), '| 사용자당 20개 추천')
    print('Kaggle → Submit Predictions → submission.csv 선택해서 제출하세요.')
    display(pd.read_csv(submission).head(3))

## 10. 결과 파일 내려받기

오른쪽 **Output → `/kaggle/working/submission.csv`**에서 제출 파일을 내려받을 수 있습니다. 추가로 `ablation_top20/ablation_results.csv`, 로그와 설정을 받아 두면 팀원에게 결과를 설명하거나 실험을 재현할 수 있습니다.

**중요:** `Save Version → Save & Run All`은 모든 실험을 다시 수행할 수 있으니, 재실행 설정을 확인하세요. 아래 셀은 파일이 있을 때만 다운로드 링크를 표시합니다.

In [ ]:
from IPython.display import FileLink, display
for path in (WORK/'submission.csv', OUT/'ablation_results.csv',
             OUT/'selected_submission_config.json'):
    if path.exists():
        display(FileLink(str(path)))
    else:
        print('아직 없음:', path)